### The rendered_prompt returned by this function is now ready to use as the input for the model you chose!

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")


In [4]:
messages = [
    {"role": "system", "content": "You are an AI assistant with access to various tools."},
    {"role": "user", "content": "Hi !"},
    {"role": "assistant", "content": "Hi human, what can help you with ?"},
]
rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [5]:
rendered_prompt

'<|im_start|>system\nYou are an AI assistant with access to various tools.<|im_end|>\n<|im_start|>user\nHi !<|im_end|>\n<|im_start|>assistant\nHi human, what can help you with ?<|im_end|>\n<|im_start|>assistant\n'

### What are Tools?

In [9]:
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b
x = calculator(2,3)
# print(calculator.to_string())

6


#### Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int

In [13]:
class Tool:
    """
    A class representing a reusable piece of code (Tool).
    
    Attributes:
        name (str): Name of the tool.
        description (str): A textual description of what the tool does.
        func (callable): The function this tool wraps.
        arguments (list): A list of argument.
        outputs (str or list): The return type(s) of the wrapped function.
    """
    def __init__(self, 
                 name: str, 
                 description: str, 
                 func: callable, 
                 arguments: list,
                 outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        """
        Return a string representation of the tool, 
        including its name, description, arguments, and outputs.
        """
        args_str = ", ".join([
            f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments
        ])
        
        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        """
        Invoke the underlying function (callable) with provided arguments.
        """
        return self.func(*args, **kwargs)

In [15]:
calculator_tool = Tool(
    "calculator",                   # name
    "Multiply two integers.",       # description
    calculator,                     # function to call
    [(3, "int"), (3, "int")],   # inputs (names and types)
    "int",                          # output
)

In [17]:
calculator_tool

In [18]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")

# Define the reasoning process
messages = [
    {"role": "system", "content": "You are an AI assistant with access to various tools."},
    {"role": "user", "content": "Calculate the product of 2 and 3."},
    {"role": "assistant", "content": "Thought: I need to multiply two numbers to get the product."},
    {"role": "assistant", "content": "Reason: The user asked for the product of 2 and 3, which means I should perform a multiplication."},
    {"role": "assistant", "content": "Action: calculator(2, 3)"},
    {"role": "assistant", "content": "Observation: The result of the multiplication is 6."},
    {"role": "assistant", "content": "Final Answer: The product of 2 and 3 is 6."}
]

# Render the prompt
rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Define the calculator function
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

# Tool class to wrap functions as tools
class Tool:
    """
    A class representing a reusable piece of code (Tool).
    
    Attributes:
        name (str): Name of the tool.
        description (str): A textual description of what the tool does.
        func (callable): The function this tool wraps.
        arguments (list): A list of argument.
        outputs (str or list): The return type(s) of the wrapped function.
    """
    def __init__(self, 
                 name: str, 
                 description: str, 
                 func: callable, 
                 arguments: list,
                 outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        """
        Return a string representation of the tool, 
        including its name, description, arguments, and outputs.
        """
        args_str = ", ".join([
            f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments
        ])
        
        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        """
        Invoke the underlying function (callable) with provided arguments.
        """
        return self.func(*args, **kwargs)

# Register the calculator as a tool
multiply_tool = Tool(
    name="calculator",
    description="Multiply two integers.",
    func=calculator,
    arguments=[("a", "int"), ("b", "int")],
    outputs="int"
)

# Example usage
result = multiply_tool(2, 3)
print(f"Result: {result}")  # Should print: Result: 6
print(multiply_tool.to_string())


Result: 6
Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
